<a href="https://colab.research.google.com/github/SutthipongS/BBL-AI-Engineer-Programming-Test-Agentic-AI-SutthipongS/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install langgraph langchain-core langchain-google-genai

In [3]:
import google.generativeai as genai
from google.colab import userdata

# ตั้งค่า API Key
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

# ปริ้นท์ชื่อโมเดลทั้งหมดที่ใช้สร้าง Text ได้
print("=== ชื่อโมเดลที่ใช้งานได้ ===")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


=== ชื่อโมเดลที่ใช้งานได้ ===
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemi

In [4]:
from google.colab import userdata
import os
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
# ==========================================
# 0. Auto-Create Knowledge Base File
# ==========================================
def create_knowledge_base():
    """สร้างไฟล์ knowledge_base.txt อัตโนมัติหากยังไม่มี"""
    content = """[Company Policy: International Travel]
All employees traveling internationally for business purposes must submit a travel request form at least 14 days prior to departure. Flights must be booked in economy class for any flight duration under 6 hours. For flights exceeding 6 hours, premium economy is permissible with prior managerial approval.

[Company Policy: Travel Expenses and Per Diem]
The daily per diem for international travel is set at $150, which covers meals and incidental expenses. Accommodation is capped at $200 per night. All receipts must be retained and submitted via the internal reimbursement portal within 7 days of returning.

[Company Policy: Remote Work]
Employees are allowed to work remotely for up to two days a week. Core working hours of 10:00 AM to 3:00 PM must be strictly observed regardless of the work location."""

    with open("knowledge_base.txt", "w", encoding="utf-8") as f:
        f.write(content)
    print("[System] knowledge_base.txt has been generated/verified.\n")

# ==========================================
# 1. Define the State
# ==========================================
class AgentState(TypedDict):
    query: str
    retrieved_snippets: str
    final_report: str

# ==========================================
# 2. Custom Tool: Data Retrieval (RAG)
# ==========================================
def simple_retriever_tool(query: str) -> str:
    """อ่านไฟล์ knowledge_base.txt และค้นหาด้วย Keyword"""
    with open("knowledge_base.txt", "r", encoding="utf-8") as f:
        paragraphs = f.read().split('\n\n')

    # ดึงคำค้นหาโดยตัดคำสั้นๆ และเครื่องหมายวรรคตอนออก
    keywords = [word.lower() for word in query.replace('?', '').split() if len(word) > 3]

    relevant_snippets = []
    for p in paragraphs:
        if any(kw in p.lower() for kw in keywords):
            relevant_snippets.append(p.strip())

    if not relevant_snippets:
        return "No relevant information found in the knowledge base."

    return "\n\n---\n\n".join(relevant_snippets)

# ==========================================
# 3. Agent 1: Data Retriever Node
# ==========================================
def data_retriever_agent(state: AgentState):
    query = state["query"]
    print(f"--- [Agent 1: Data Retriever] Analyzing query: '{query}' ---")

    snippets = simple_retriever_tool(query)
    print(">> Successfully retrieved relevant text snippets.\n")

    return {"retrieved_snippets": snippets}

# ==========================================
# 4. Agent 2: Report Generator Node
# ==========================================
def report_generator_agent(state: AgentState):
    query = state["query"]
    snippets = state["retrieved_snippets"]
    print("--- [Agent 2: Report Generator] Synthesizing snippets into a cohesive report... ---")

    # ⚠️ ดึง API Key จาก Secrets ของ Colab ⚠️
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert report generator.
        Use ONLY the provided context snippets to answer the user's query.
        Synthesize the information into a cohesive, non-redundant, and well-formatted answer.
        Use bullet points if necessary. Do not add any external knowledge."""),
        ("user", "Query: {query}\n\nContext Snippets:\n{snippets}")
    ])

    chain = prompt | llm
    response = chain.invoke({"query": query, "snippets": snippets})

    return {"final_report": response.content}

# ==========================================
# 5. Task Orchestration (LangGraph)
# ==========================================
def build_workflow():
    workflow = StateGraph(AgentState)
    workflow.add_node("Data_Retriever", data_retriever_agent)
    workflow.add_node("Report_Generator", report_generator_agent)

    workflow.set_entry_point("Data_Retriever")
    workflow.add_edge("Data_Retriever", "Report_Generator")
    workflow.add_edge("Report_Generator", END)

    return workflow.compile()

# ==========================================
# 6. Execution and Evaluation
# ==========================================
if __name__ == "__main__":
    # 1. สร้างไฟล์ Text อัตโนมัติ
    create_knowledge_base()

    # 2. สร้าง Graph Workflow
    app = build_workflow()

    # 3. รันทดสอบ 2 คำถามตามที่โจทย์แนะนำ
    test_queries = [
        "What is the policy on international travel?",
        "How much is the daily per diem and accommodation limit?"
    ]

    for q in test_queries:
        print("="*70)
        print(f"USER QUERY: {q}")
        print("="*70)

        # รัน Agent
        result = app.invoke({"query": q})

        print("\n" + "="*70)
        print("FINAL OUTPUT:")
        print("="*70)
        print(result["final_report"])
        print("\n\n")

[System] knowledge_base.txt has been generated/verified.

USER QUERY: What is the policy on international travel?
--- [Agent 1: Data Retriever] Analyzing query: 'What is the policy on international travel?' ---
>> Successfully retrieved relevant text snippets.

--- [Agent 2: Report Generator] Synthesizing snippets into a cohesive report... ---

FINAL OUTPUT:
The policy on international travel for business purposes includes the following:

*   **Travel Request:** Employees must submit a travel request form at least 14 days prior to departure.
*   **Flight Class:**
    *   Flights under 6 hours must be booked in economy class.
    *   For flights exceeding 6 hours, premium economy is permissible with prior managerial approval.
*   **Per Diem:** The daily per diem is $150, covering meals and incidental expenses.
*   **Accommodation:** Accommodation is capped at $200 per night.
*   **Reimbursement:** All receipts must be retained and submitted via the internal reimbursement portal within 7